# Partners Analytics Pipeline (Ch. 1-17)

This notebook follows the full analytics lifecycle for `partners.csv` and includes:

1. Problem framing (business objective + success metrics + prediction vs explanation)
2. Data acquisition and preparation (reproducible transformations)
3. Exploration (distributions, anomalies, relationships)
4. Modeling (explanatory and predictive models)
5. Evaluation and selection (metrics + fairness checks)
6. Feature selection (importance + rationale)
7. Deployment outputs (saved model + schema + top features + recommendation notes)

The pipeline writes artifacts to `ml-pipelines/artifacts/`.

In [3]:
# Phase 1: Problem Framing
# Business question:
#   Which partners are at highest risk of becoming inactive, and what actions reduce this risk?
#
# Modeling goals:
# 1) Predictive goal: classify whether partner will be inactive (operational early warning).
# 2) Explanatory goal: estimate direction/strength of relationships between partner attributes and inactivity risk.
#
# Success metrics:
# - Predictive: ROC-AUC, F1 (minority class), recall for inactive class.
# - Explanatory: stable, interpretable coefficients and signs.
# - Business: identify actionable risk segments by region/role/type.

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
import joblib

RANDOM_STATE = 42

# Make paths robust whether notebook is run from project root or ml-pipelines/
candidate_artifact_dirs = [Path('ml-pipelines/artifacts'), Path('artifacts')]
ARTIFACT_DIR = next((p for p in candidate_artifact_dirs if p.parent.exists()), candidate_artifact_dirs[0])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

candidate_data_paths = [
    Path('datasets/partners.csv'),
    Path('../datasets/partners.csv'),
]
DATA_PATH = next((p for p in candidate_data_paths if p.exists()), candidate_data_paths[0])
assert DATA_PATH.exists(), f'Missing dataset. Checked: {[str(p) for p in candidate_data_paths]}'

In [4]:
# Phase 2: Data Acquisition and Preparation (reproducible)
raw_df = pd.read_csv(DATA_PATH)

def prepare_partners(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['start_date'] = pd.to_datetime(out['start_date'], errors='coerce')
    out['end_date'] = pd.to_datetime(out['end_date'], errors='coerce')

    reference_date = pd.Timestamp('2025-12-31')
    out['inactive_flag'] = (out['status'].str.lower() == 'inactive').astype(int)

    # If still active, tenure is measured up to reference date
    out['effective_end_date'] = out['end_date'].fillna(reference_date)
    out['tenure_days'] = (out['effective_end_date'] - out['start_date']).dt.days

    # Simple quality/engagement proxies from available columns
    out['has_notes'] = out['notes'].fillna('').str.strip().ne('').astype(int)
    out['email_domain'] = out['email'].str.split('@').str[-1].str.lower().fillna('unknown')

    # Binary indicator for secondary contractor role in notes
    out['secondary_contractor_flag'] = out['notes'].fillna('').str.contains('Secondary', case=False).astype(int)

    # Keep clean modeling table
    keep_cols = [
        'partner_type', 'role_type', 'region', 'has_notes', 'email_domain',
        'secondary_contractor_flag', 'tenure_days', 'inactive_flag'
    ]
    return out[keep_cols]

partners_df = prepare_partners(raw_df)
partners_df.head()

,partner_type,role_type,region,has_notes,email_domain,secondary_contractor_flag,tenure_days,inactive_flag
0,Organization,SafehouseOps,Luzon,1,hopepartners.ph,0,1460,0
1,Individual,Evaluation,Luzon,1,pldt.net.ph,0,1440,0
2,Individual,Education,Mindanao,1,eastern.com.ph,0,1420,0
3,Organization,Logistics,Visayas,1,bayanihanfoundation.ph,0,1400,0
4,Individual,SafehouseOps,Visayas,1,yahoo.com.ph,0,1380,0


In [5]:
# Persist prepared data for reproducibility
prepared_path = ARTIFACT_DIR / 'partners_prepared.csv'
partners_df.to_csv(prepared_path, index=False)
prepared_path

WindowsPath('ml-pipelines/artifacts/partners_prepared.csv')

In [6]:
# Phase 3: Exploration
print('Shape:', partners_df.shape)
print('\nMissing values:')
print(partners_df.isna().sum())

print('\nTarget distribution (inactive_flag):')
print(partners_df['inactive_flag'].value_counts(dropna=False))

print('\nCategorical distributions:')
for col in ['partner_type', 'role_type', 'region', 'email_domain']:
    print(f'\n{col}:')
    print(partners_df[col].value_counts())

print('\nTenure summary by status:')
print(partners_df.groupby('inactive_flag')['tenure_days'].describe())

Shape: (30, 8)

Missing values:
partner_type                 0
role_type                    0
region                       0
has_notes                    0
email_domain                 0
secondary_contractor_flag    0
tenure_days                  0
inactive_flag                0
dtype: int64

Target distribution (inactive_flag):
inactive_flag
0    27
1     3
Name: count, dtype: int64

Categorical distributions:

partner_type:
partner_type
Individual      20
Organization    10
Name: count, dtype: int64

role_type:
role_type
SafehouseOps     8
Education        6
Logistics        5
Maintenance      5
Evaluation       4
FindSafehouse    1
Transport        1
Name: count, dtype: int64

region:
region
Luzon       11
Visayas     10
Mindanao     9
Name: count, dtype: int64

email_domain:
email_domain
eastern.com.ph            7
yahoo.com.ph              4
smart.com.ph              4
globe.com.ph              3
hopepartners.ph           2
pldt.net.ph               2
bayanihanfoundation.ph    1
u

In [7]:
# Phase 4: Modeling setup (explanatory + predictive)
X = partners_df.drop(columns=['inactive_flag'])
y = partners_df['inactive_flag']

categorical_features = ['partner_type', 'role_type', 'region', 'email_domain']
numeric_features = ['has_notes', 'secondary_contractor_flag', 'tenure_days']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
    ]
)

# Explanatory model: regularized logistic regression (interpretable coefficient signs/magnitudes)
explanatory_model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE)),
])

# Predictive model: random forest (captures non-linear interactions)
predictive_model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
    )),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train size:', X_train.shape, '| Test size:', X_test.shape)

Train size: (21, 7) | Test size: (9, 7)


In [8]:
# Phase 5: Evaluation and Selection (CV + holdout + fairness)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}

cv_explanatory = cross_validate(explanatory_model, X_train, y_train, cv=cv, scoring=scoring)
cv_predictive = cross_validate(predictive_model, X_train, y_train, cv=cv, scoring=scoring)

print('Explanatory CV means:')
for metric in scoring:
    print(f"  {metric}: {cv_explanatory['test_' + metric].mean():.3f}")

print('\nPredictive CV means:')
for metric in scoring:
    print(f"  {metric}: {cv_predictive['test_' + metric].mean():.3f}")

# Fit both models for comparison
explanatory_model.fit(X_train, y_train)
predictive_model.fit(X_train, y_train)

# Choose production model by ROC-AUC (on holdout)
exp_proba = explanatory_model.predict_proba(X_test)[:, 1]
pred_proba = predictive_model.predict_proba(X_test)[:, 1]

exp_auc = roc_auc_score(y_test, exp_proba)
pred_auc = roc_auc_score(y_test, pred_proba)

best_model_name = 'predictive_model' if pred_auc >= exp_auc else 'explanatory_model'
best_model = predictive_model if best_model_name == 'predictive_model' else explanatory_model

print(f'\nHoldout ROC-AUC explanatory: {exp_auc:.3f}')
print(f'Holdout ROC-AUC predictive:  {pred_auc:.3f}')
print(f'\nSelected production model: {best_model_name}')

# Business-focused evaluation on selected model
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('\nClassification report (selected model):')
print(classification_report(y_test, y_pred, digits=3, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

# Fairness slice check by partner_type and region
fairness_rows = []
y_proba_series = pd.Series(y_proba, index=X_test.index)

for group_col in ['partner_type', 'region']:
    grouped = X_test.copy()
    grouped['y_true'] = y_test
    grouped['y_pred'] = y_pred
    for g, sub in grouped.groupby(group_col):
        if sub['y_true'].nunique() < 2:
            group_auc = np.nan
        else:
            group_auc = roc_auc_score(sub['y_true'], y_proba_series.loc[sub.index])
        fairness_rows.append({
            'group_column': group_col,
            'group_value': g,
            'n': len(sub),
            'positive_rate_pred': sub['y_pred'].mean(),
            'recall': recall_score(sub['y_true'], sub['y_pred'], zero_division=0),
            'precision': precision_score(sub['y_true'], sub['y_pred'], zero_division=0),
            'auc': group_auc,
        })

fairness_df = pd.DataFrame(fairness_rows)
fairness_df

c:\Users\abiga\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
c:\Users\abiga\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\abiga\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abiga\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples

Explanatory CV means:
  roc_auc: nan
  f1: 0.000
  recall: 0.000
  precision: 0.000

Predictive CV means:
  roc_auc: nan
  f1: 0.000
  recall: 0.000
  precision: 0.000

Holdout ROC-AUC explanatory: 0.875
Holdout ROC-AUC predictive:  0.875

Selected production model: predictive_model

Classification report (selected model):
              precision    recall  f1-score   support

           0      0.889     1.000     0.941         8
           1      0.000     0.000     0.000         1

    accuracy                          0.889         9
   macro avg      0.444     0.500     0.471         9
weighted avg      0.790     0.889     0.837         9

Confusion matrix:
[[8 0]
 [1 0]]


,group_column,group_value,n,positive_rate_pred,recall,precision,auc
0,partner_type,Individual,6,0.0,0.0,0.0,NaN
1,partner_type,Organization,3,0.0,0.0,0.0,1.0
2,region,Luzon,4,0.0,0.0,0.0,NaN
3,region,Mindanao,2,0.0,0.0,0.0,NaN
4,region,Visayas,3,0.0,0.0,0.0,1.0


In [9]:
# Phase 6: Feature Selection and Impact
# Explanatory coefficients
exp_clf = explanatory_model.named_steps['clf']
exp_feature_names = explanatory_model.named_steps['prep'].get_feature_names_out()
exp_coeff = pd.DataFrame({
    'feature': exp_feature_names,
    'coefficient': exp_clf.coef_[0],
}).sort_values('coefficient', key=np.abs, ascending=False)

# Predictive feature importances
pred_clf = predictive_model.named_steps['clf']
pred_feature_names = predictive_model.named_steps['prep'].get_feature_names_out()
pred_importance = pd.DataFrame({
    'feature': pred_feature_names,
    'importance': pred_clf.feature_importances_,
}).sort_values('importance', ascending=False)

top_features = pred_importance.head(15).copy()
top_features_path = ARTIFACT_DIR / 'partners_top_features.csv'
top_features.to_csv(top_features_path, index=False)

print('Top predictive features:')
display(top_features)

print('\nTop explanatory coefficients (absolute magnitude):')
display(exp_coeff.head(15))

Top predictive features:


,feature,importance
23,num__tenure_days,0.247734
2,cat__role_type_Education,0.101507
8,cat__region_Mindanao,0.082775
12,cat__email_domain_eastern.com.ph,0.073828
19,cat__email_domain_smart.com.ph,0.069406
22,num__secondary_contractor_flag,0.067538
7,cat__region_Luzon,0.062968
6,cat__role_type_SafehouseOps,0.055478
4,cat__role_type_Logistics,0.049105
9,cat__region_Visayas,0.048753



Top explanatory coefficients (absolute magnitude):


,feature,coefficient
23,num__tenure_days,-1.646900
4,cat__role_type_Logistics,0.494039
5,cat__role_type_Maintenance,-0.491929
19,cat__email_domain_smart.com.ph,0.463437
9,cat__region_Visayas,-0.422386
2,cat__role_type_Education,0.410156
1,cat__partner_type_Organization,-0.311267
0,cat__partner_type_Individual,0.310771
6,cat__role_type_SafehouseOps,-0.307869
8,cat__region_Mindanao,0.301401


In [10]:
# Phase 7: Deployment Artifacts
model_path = ARTIFACT_DIR / 'partners_predictive_model.joblib'
joblib.dump(best_model, model_path)

schema = {
    'dataset': 'partners.csv',
    'target': 'inactive_flag',
    'selected_model': best_model_name,
    'features': X.columns.tolist(),
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'metrics_holdout': {
        'roc_auc': float(roc_auc_score(y_test, y_proba)),
        'f1': float(f1_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
    },
}

schema_path = ARTIFACT_DIR / 'partners_model_schema.json'
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2)

fairness_path = ARTIFACT_DIR / 'partners_fairness_report.csv'
fairness_df.to_csv(fairness_path, index=False)

# Decision recommendations from feature evidence
recommendations = [
    'Prioritize retention check-ins for partner segments with highest predicted inactivity risk.',
    'Design region-specific partner support where fairness slices show weaker recall.',
    'Audit onboarding/engagement practices by role_type where risk-driving features concentrate.',
    'Use explanatory coefficients to communicate transparent risk factors to leadership.',
]

recommendation_path = ARTIFACT_DIR / 'partners_recommendations.txt'
recommendation_path.write_text('\n'.join(f'- {r}' for r in recommendations), encoding='utf-8')

print('Artifacts written:')
print('-', model_path)
print('-', schema_path)
print('-', fairness_path)
print('-', top_features_path)
print('-', recommendation_path)

Artifacts written:
- ml-pipelines\artifacts\partners_predictive_model.joblib
- ml-pipelines\artifacts\partners_model_schema.json
- ml-pipelines\artifacts\partners_fairness_report.csv
- ml-pipelines\artifacts\partners_top_features.csv
- ml-pipelines\artifacts\partners_recommendations.txt


## Interpretation for Stakeholders

- **Predictive pipeline use:** Score partners monthly and flag high-risk records for proactive outreach.
- **Explanatory pipeline use:** Use coefficient directions to explain *why* risk is elevated by partner segment.
- **Feature impact:** The exported `partners_top_features.csv` lists the most influential predictors in the production model.
- **Decision support:** Focus retention resources on high-risk segments, then monitor performance/fairness by `partner_type` and `region`.
- **Caution:** This dataset is small and class-imbalanced, so treat this as a baseline and retrain with more records over time.